# 00 · 검증 도구 점검

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_llm_dit/blob/main/notebooks/00_verify.ipynb)

이후 모든 블록 구현이 통과해야 하는 **판정 기준**을 먼저 점검하는 노트북입니다.

1. 환경 준비
2. 출력 비교 도구 자체 점검: 같은 함수면 PASS, 다르면 FAIL이 나오는지
3. 자체 일관성 검사 (causal mask 누수)
4. HF 원본 구조 읽기 (가중치 다운로드 없이 헤더만)
5. 구조(shape) 대조 도구 점검
6. 단계별 검증 현황표
7. 결과 저장

**런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 위에서부터 실행하세요.

## 1. 환경 준비

In [1]:
import os
REPO_DIR = "/content/tmp_llm_dit"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/HisameOgasahara/tmp_llm_dit.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip -q install -U -r requirements-colab.txt

/content/tmp_llm_dit
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 71.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have 

In [2]:
import sys, torch, transformers, diffusers
print("python      ", sys.version.split()[0])
print("torch       ", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
print("transformers", transformers.__version__)
print("diffusers   ", diffusers.__version__)
!git log -1 --format='repo commit: %h %s'

python       3.13.15
torch        2.11.0+cu130 | CUDA: True | Tesla T4
transformers 5.19.0
diffusers    0.41.0
repo commit: a4ba106 Add verification toolkit and 00_verify notebook


In [3]:
# gated 모델(Krea-2 등) 헤더를 읽을 때만 필요. Colab 왼쪽 🔑(보안 비밀)에 HF_TOKEN 등록.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN 설정됨")
except Exception:
    print("HF_TOKEN 없음 (공개 모델만 확인 가능)")

HF_TOKEN 설정됨


## 2. 출력 비교 도구 자체 점검

원본 예시로 HF GPT-2를 씁니다.
- **같은 구조**인 별도 모델에 가중치를 복사하면 → ✅ PASS가 나와야 합니다.
- **일부러 수식을 바꾼** 모델(LayerNorm eps 변경)은 → ❌ FAIL이 나와야 합니다.
- `hook_pairs`로 층별 중간 출력도 비교해서, 어느 층부터 어긋나는지 보여줍니다.

In [4]:
from transformers import GPT2Config, GPT2Model
from miniarch import verify as V

V.set_deterministic(0)
ref, inputs = V.cases.ref_gpt2()

same = GPT2Model(ref.config)                                  # 같은 구조, 다른 랜덤 초기화
cfg_bad = GPT2Config(**{**ref.config.to_dict(), "layer_norm_epsilon": 1e-1})
bad = GPT2Model(cfg_bad)                                      # 수식이 살짝 다름

layers = [("h.0", "h.0"), ("h.1", "h.1")]
print("[같은 구조]");    print(V.check_equivalence(ref, same, inputs, hook_pairs=layers).summary())
print("\n[다른 수식]"); print(V.check_equivalence(ref, bad,  inputs, hook_pairs=layers).summary())

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 127), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 127), got 50256. This may result in unexpected behavior.


[같은 구조]
✅ PASS  (복사 28개)
  ✅ h.0  <->  h.0                            max_abs=0.00e+00 
  ✅ h.1  <->  h.1                            max_abs=0.00e+00 
  ✅ [final output]                           max_abs=0.00e+00 

[다른 수식]
❌ FAIL  (복사 28개)
  ❌ h.0  <->  h.0                            max_abs=5.48e-02 
  ❌ h.1  <->  h.1                            max_abs=7.47e-02 
  ❌ [final output]                           max_abs=3.32e+00 


## 3. 자체 일관성 검사: causal mask 누수

뒤쪽 토큰을 바꿨을 때 앞쪽 출력이 변하면 미래 정보가 새고 있다는 뜻입니다.
HF Llama(정상)는 PASS, mask를 빼고 양방향으로 본 경우는 FAIL이 나와야 합니다.

In [5]:
dev = "cuda" if torch.cuda.is_available() else "cpu"
llama, _ = V.cases.ref_llama()
llama = llama.to(dev).eval(); V.force_eager(llama)

causal = lambda x: llama(input_ids=x).last_hidden_state
# 모든 위치를 볼 수 있는 4D mask(0 = 허용)를 넘겨 일부러 양방향으로 만든 경우
bidir  = lambda x: llama(input_ids=x, attention_mask=torch.zeros(x.shape[0], 1, x.shape[1], x.shape[1], device=x.device)).last_hidden_state

for name, f in [("Llama causal", causal), ("Llama 양방향(누수)", bidir)]:
    try:
        r = V.causal_leak_test(f, vocab_size=V.cases.VOCAB, device=dev)
        print(f"{'✅' if r.ok else '❌'} {name:<18} max_abs={r.max_abs:.2e}")
    except Exception as e:
        print(f"⚠️ {name}: {type(e).__name__}: {e}")

✅ Llama causal       max_abs=0.00e+00
❌ Llama 양방향(누수)      max_abs=1.37e+00


## 4. HF 원본 구조 읽기 (헤더만, 다운로드 없음)

safetensors 헤더를 HTTP range 요청으로 읽어서 실제 가중치 이름, shape, dtype을 확인합니다.
routed expert는 개수가 많아 숨깁니다.

In [6]:
def peek(repo, pattern, subfolder="", limit=60):
    print(f"\n===== {repo}  /{pattern}/")
    try:
        cfg = V.fetch_config(repo, subfolder)
        cfg = cfg.get("text_config", cfg)
        keys = ["model_type", "hidden_size", "num_hidden_layers", "num_attention_heads", "num_key_value_heads", "head_dim"]
        print({k: cfg[k] for k in keys if k in cfg})
        h = V.read_model_header(repo, subfolder=subfolder, pattern=pattern)
        h = {k: v for k, v in h.items() if ".experts." not in k}
        V.show(h, limit)
    except Exception as e:
        print(f"⚠️ {type(e).__name__}: {e}")

peek("deepseek-ai/DeepSeek-V4-Flash", r"layers\.0\.")              # MLA(V4) + mHC + sparse
peek("Qwen/Qwen3.6-27B",             r"layers\.(0|3)\.")          # Gated DeltaNet(0) / Gated Attn(3)
peek("moonshotai/Kimi-K3",           r"layers\.(1|4)\.")          # KDA(1) / MLA(4)
peek("Qwen/Qwen-Image-2.1",          r"blocks\.0\.|transformer_blocks\.0\.", subfolder="transformer")


===== deepseek-ai/DeepSeek-V4-Flash  /layers\.0\./
{'model_type': 'deepseek_v4', 'hidden_size': 4096, 'num_hidden_layers': 43, 'num_attention_heads': 64, 'num_key_value_heads': 1, 'head_dim': 512}
layers.0.attn.attn_sink                                                F32      [64]
layers.0.attn.kv_norm.weight                                           BF16     [512]
layers.0.attn.q_norm.weight                                            BF16     [1024]
layers.0.attn.wkv.scale                                                F8_E8M0  [4, 32]
layers.0.attn.wkv.weight                                               F8_E4M3  [512, 4096]
layers.0.attn.wo_a.scale                                               F8_E8M0  [64, 32]
layers.0.attn.wo_a.weight                                              F8_E4M3  [8192, 4096]
layers.0.attn.wo_b.scale                                               F8_E8M0  [32, 64]
layers.0.attn.wo_b.weight                                              F8_E4M3  [4096, 8192]


## 5. 구조(shape) 대조 도구 점검

원본 config로 **meta device**(메모리 사용 0)에 모델을 만들고, 그 state_dict를 원본 헤더와 대조합니다.
여기서는 HF의 Qwen3 구현과 실제 Qwen3-0.6B 체크포인트를 비교하니 ✅가 나와야 정상입니다.
나중에는 HF 구현 자리에 **내 구현**을 넣어서 같은 방식으로 검사합니다.

In [7]:
from transformers import AutoConfig, AutoModelForCausalLM

repo = "Qwen/Qwen3-0.6B"
cfg = AutoConfig.from_pretrained(repo)
mine_shapes = V.meta_state_shapes(lambda: AutoModelForCausalLM.from_config(cfg))
orig = V.read_model_header(repo)
print(V.compare_shapes(mine_shapes, orig, scope=r"layers\.0\.|embed_tokens|model\.norm").summary())

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

✅ PASS  일치 13개


## 6. 단계별 검증 현황표

- 원본 builder가 현재 라이브러리 버전에서 동작하는지 먼저 확인합니다.
- 그다음 전체 표를 출력합니다. 지금은 내 구현이 없어서 ⬜ 상태가 정상이고, 구현을 추가할 때마다 ✅로 바뀝니다.

In [8]:
V.cases.smoke_refs()

✅ A0   gpt2     params=112,384  out=(2, 16, 64)
✅ A1   Llama    params=82,240  out=(2, 16, 64)
❌ B0   DiT      RuntimeError: Failed to import diffusers.models.transformers.dit_transformer_2d because of the following error (look up to see its traceback):
Failed to import diffusers.loaders.peft because of the following error (look up to see its traceback):
cannot import name 'FqnToConfig' from 'torchao.quantization' (/usr/local/lib/python3.13/dist-packages/torchao/quantization/__init__.py)


In [9]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)
status = V.cases.run_all()
status[["stage", "block", "기준", "status"]]

,stage,block,기준,status
0,A0,"GPT-2 (MHA, learned pos, LayerNorm, GELU)",gpt2,⬜ 미구현
1,A1,"Llama (RoPE, RMSNorm, SwiGLU, GQA)",Llama,⬜ 미구현
2,A2,QK-norm / SWA+full / sink / partial RoPE,MiMo-V2.6,⬜ 원본 builder 미작성
3,A3,"MoE (fine-grained + shared, aux-free)",DeepSeek-V4 · GLM-5.3,⬜ 원본 builder 미작성
4,A4,"MLA (V3 → V4: 1 KV head, o_lora)",DeepSeek-V4,⬜ 원본 builder 미작성
5,A5a,DSA indexer (+층 간 공유),GLM-5.3,⬜ 원본 builder 미작성
6,A5b,블록 top-k sparse (MSA),MiniMax-M3,⬜ 원본 builder 미작성
7,A6a,Gated DeltaNet + Gated Attention,Qwen3.6/3.8-27B,⬜ 원본 builder 미작성
8,A6b,KDA + gated MLA,Kimi-K3,⬜ 원본 builder 미작성
9,A7a,Hyper-Connections / mHC,DeepSeek-V4 · Qwen3.8-Flash-Next,⬜ 원본 builder 미작성


In [10]:
# FAIL / 에러 상세
for _, r in status[status.status.str.contains("FAIL|에러|실패")].iterrows():
    print(f"--- {r.stage} {r.block}\n{r.detail}\n")

--- B0 DiT (adaLN-Zero)
RuntimeError: Failed to import diffusers.models.transformers.dit_transformer_2d because of the following error (look up to see its traceback):
Failed to import diffusers.loaders.peft because of the following error (look up to see its traceback):
cannot import name 'FqnToConfig' from 'torchao.quantization' (/usr/local/lib/python3.13/dist-packages/torchao/quantization/__init__.py)



## 7. 결과 저장

- `results/`에 현황표를 저장합니다.
- 노트북 출력까지 남기려면 **파일 → GitHub에 사본 저장**으로 이 저장소의 `notebooks/`에 저장하세요.
- Drive에도 남기려면 아래 셀의 `SAVE_TO_DRIVE = True`로 바꾸세요.

In [11]:
import datetime, shutil
stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M")
os.makedirs("results", exist_ok=True)
path = f"results/00_verify_{stamp}.csv"
status.to_csv(path, index=False)
print("saved:", path)

SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dst = "/content/drive/MyDrive/tmp_llm_dit/results"
    os.makedirs(dst, exist_ok=True)
    shutil.copy(path, dst)
    print("drive:", dst)

saved: results/00_verify_20261014-0531.csv
